# LeetCode 1059 — All Paths from Source Lead to Destination bằng Python

Link đề: https://leetcode.com/problems/all-paths-from-source-lead-to-destination/ · Độ khó: **Medium** · Premium (cần tài khoản LeetCode Premium để xem đề) · Chủ đề: Graph (đồ thị), DFS, Topological Sort (sắp xếp tô-pô), phát hiện chu trình

> **Lưu ý:** đề bài của bài Premium này được viết theo hiểu biết và theo bản mô tả đã có; nên đối chiếu lại nếu bạn có tài khoản Premium.

Skeleton LeetCode:

```python
class Solution:
    def leadsToDestination(self, n: int, edges: List[List[int]], source: int, destination: int) -> bool:
        ...
```

Ghi chú: cho đồ thị **có hướng** gồm `n` đỉnh và danh sách cạnh `edges`, cùng hai đỉnh `source` và `destination`. Hỏi: **mọi** đường đi xuất phát từ `source` có cuối cùng đều kết thúc tại `destination` không? Cụ thể cần cả ba điều: (1) có ít nhất một đường từ `source` đến `destination`; (2) mọi đường từ `source` tới một đỉnh **không có cạnh đi ra** thì đỉnh đó phải là `destination`; (3) số đường đi từ `source` là **hữu hạn** (không đi vào chu trình). Ý tưởng chính: **DFS ba trạng thái** (chưa thăm / đang nằm trên đường đi hiện tại / đã xong), gặp lại đỉnh đang trên đường đi là có chu trình; gặp đỉnh không có cạnh ra thì phải bằng `destination`.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: ba điều kiện (có đường tới đích, mọi "ngõ cụt" phải là đích, không có chu trình vô hạn) và dạng gọn **"không có chu trình đạt được từ source, và mọi đỉnh đạt được không có cạnh ra đều là `destination`"**.
2. Nắm các công cụ Python cần thiết: danh sách kề `graph[u]`, mảng trạng thái, ngăn xếp tự quản lý (DFS không đệ quy), `collections.deque` cho BFS/Kahn, `sys.setrecursionlimit` và vì sao nên tránh đệ quy sâu.
3. Hiểu **ba trạng thái** của DFS: `0` chưa thăm, `1` đang nằm trên đường đi hiện tại (nếu gặp lại là **chu trình**), `2` đã xong (đã chứng minh mọi đường từ đỉnh này dẫn tới `destination`).
4. Đi qua 4 cách: liệt kê mọi đường đi có giới hạn độ dài (brute force, dùng làm oracle), DFS đệ quy ba trạng thái, DFS lặp bằng ngăn xếp (cách nộp chính) và thuật toán Kahn (sắp xếp tô-pô) trên phần đồ thị đạt được từ `source`.
5. Tránh các lỗi kinh điển: chỉ kiểm tra đến được `destination`, quên chu trình, dừng ngay khi tới `destination`, dùng một mảng `visited` duy nhất (nhầm đường đi kim cương với chu trình), duyệt từ mọi đỉnh thay vì chỉ từ `source`, và đệ quy quá sâu.

## 2. Đề bài & ví dụ

Cho các cạnh `edges` của một đồ thị có hướng, `edges[i] = [a_i, b_i]` là cạnh đi từ `a_i` đến `b_i`, và hai đỉnh `source`, `destination`. Hãy xác định liệu **mọi** đường đi bắt đầu từ `source` có cuối cùng kết thúc tại `destination` hay không, nghĩa là:

- Tồn tại ít nhất một đường đi từ `source` đến `destination`.
- Nếu tồn tại đường đi từ `source` đến một đỉnh không có cạnh đi ra thì đỉnh đó phải là `destination`.
- Số đường đi có thể từ `source` đến `destination` là hữu hạn.

Trả về `true` khi và chỉ khi mọi đường từ `source` đều dẫn đến `destination`.

**Ví dụ 1:** `n = 3`, `edges = [[0,1],[0,2]]`, `source = 0`, `destination = 2` -> `false`. Có thể đến và bị kẹt ở cả đỉnh `1` và đỉnh `2`.

**Ví dụ 2:** `n = 4`, `edges = [[0,1],[0,3],[1,2],[2,1]]`, `source = 0`, `destination = 3` -> `false`. Hoặc kết thúc ở `3`, hoặc lặp vô hạn quanh `1` và `2`.

**Ví dụ 3:** `n = 4`, `edges = [[0,1],[0,2],[1,3],[2,3]]`, `source = 0`, `destination = 3` -> `true`.

**Ràng buộc:**

- `1 <= n <= 10^4`
- `0 <= edges.length <= 10^4`
- `edges[i].length == 2`
- `0 <= a_i, b_i <= n - 1`, `0 <= source, destination <= n - 1`
- Đồ thị có thể có **khuyên** (cạnh nối một đỉnh với chính nó) và **cạnh song song**.

In [1]:
from typing import List
from collections import deque
import random, sys, time
from itertools import product

examples = [
    (3, [[0, 1], [0, 2]], 0, 2, False),
    (4, [[0, 1], [0, 3], [1, 2], [2, 1]], 0, 3, False),
    (4, [[0, 1], [0, 2], [1, 3], [2, 3]], 0, 3, True),
]
for n, edges, s, d, expected in examples:
    print(f"n={n} edges={edges} source={s} destination={d} -> kỳ vọng {expected}")

n=3 edges=[[0, 1], [0, 2]] source=0 destination=2 -> kỳ vọng False
n=4 edges=[[0, 1], [0, 3], [1, 2], [2, 1]] source=0 destination=3 -> kỳ vọng False
n=4 edges=[[0, 1], [0, 2], [1, 3], [2, 3]] source=0 destination=3 -> kỳ vọng True


## 3. Kiến thức nền

### 3.1. Danh sách kề

`graph[u]` là danh sách các đỉnh `v` có cạnh `u -> v`. Đỉnh **không có cạnh ra** (ngõ cụt, "sink") có `graph[u]` rỗng.

In [2]:
n, edges = 4, [[0, 1], [0, 2], [1, 3], [2, 3]]
graph = [[] for _ in range(n)]
for a, b in edges:
    graph[a].append(b)
print(graph)
print("các ngõ cụt:", [u for u in range(n) if not graph[u]])

[[1, 2], [3], [3], []]
các ngõ cụt: [3]


### 3.2. Chu trình và ba trạng thái DFS

Khi duyệt DFS, mỗi đỉnh ở một trong ba trạng thái:

- `0`: chưa thăm;
- `1`: **đang nằm trên đường đi hiện tại** (đã vào nhưng chưa ra khỏi);
- `2`: đã xong.

Nếu từ `u` ta gặp cạnh tới đỉnh `v` đang ở trạng thái `1` thì ta quay về một đỉnh trên đường đi hiện tại: **có chu trình**. Gặp đỉnh trạng thái `2` thì đã biết kết quả, không duyệt lại (đường đi "kim cương" tới cùng một đỉnh không phải chu trình).

In [3]:
# đường đi kim cương: 0 -> 1 -> 3 và 0 -> 2 -> 3: đỉnh 3 được gặp hai lần nhưng KHÔNG có chu trình
# vòng lặp: 0 -> 1 -> 2 -> 1: gặp lại đỉnh 1 đang trên đường đi hiện tại
print("kim cương: 3 được thăm hai lần nhưng ở lần thứ hai trạng thái là 2 (đã xong)")
print("vòng lặp: từ 2 đi tới 1 khi 1 còn ở trạng thái 1 (đang trên đường đi)")

kim cương: 3 được thăm hai lần nhưng ở lần thứ hai trạng thái là 2 (đã xong)
vòng lặp: từ 2 đi tới 1 khi 1 còn ở trạng thái 1 (đang trên đường đi)


### 3.3. Đệ quy sâu và ngăn xếp tự quản lý

Python giới hạn độ sâu đệ quy mặc định khoảng `1000` khung; đồ thị dạng xích dài `10^4` đỉnh làm DFS đệ quy ném `RecursionError`. Cách an toàn: **tự quản lý ngăn xếp** bằng danh sách, mỗi phần tử là đỉnh đang xử lý kèm chỉ số cạnh kế tiếp.

In [4]:
def depth_demo(k):
    return 0 if k == 0 else 1 + depth_demo(k - 1)

old = sys.getrecursionlimit()
sys.setrecursionlimit(1000)
try:
    depth_demo(5000)
    print("không lỗi (không mong đợi)")
except RecursionError as e:
    print("RecursionError:", e)
finally:
    sys.setrecursionlimit(old)

RecursionError: maximum recursion depth exceeded


## 4. Ý tưởng nền tảng / trực giác (tính tay)

Ví dụ 3: `0 -> 1`, `0 -> 2`, `1 -> 3`, `2 -> 3`, đích `3`.

- Từ `0` có hai đường `0 -> 1 -> 3` và `0 -> 2 -> 3`; cả hai kết thúc ở `3`, và `3` không có cạnh ra ⇒ `true`.

Ví dụ 1: `0 -> 1`, `0 -> 2`, đích `2`. Đỉnh `1` không có cạnh ra nhưng khác đích ⇒ có đường bị kẹt ở `1` ⇒ `false`.

Ví dụ 2: có chu trình `1 -> 2 -> 1` đạt được từ `0` ⇒ có đường đi vô hạn ⇒ `false` (dù đỉnh `3` là đích).

Dạng gọn của đề: gọi `R` là tập các đỉnh đạt được từ `source`. Câu trả lời là `true` khi và chỉ khi (a) phần đồ thị trên `R` **không có chu trình** và (b) mọi đỉnh của `R` không có cạnh ra đều bằng `destination`.

## 5. Cách 1 — Brute Force: liệt kê mọi đường đi có giới hạn độ dài

**Ý tưởng.** Liệt kê mọi đường đi (dãy đỉnh liên tiếp theo cạnh) bắt đầu từ `source`. Nếu có một đường đi dài đúng `n` cạnh thì nó đi qua `n + 1` đỉnh, nên có đỉnh lặp lại, tức có chu trình đạt được ⇒ số đường đi vô hạn ⇒ `False`. Nếu mọi đường đều ngắn hơn `n` cạnh thì kiểm tra các điều kiện: có đường kết thúc tại `destination` và mọi đường kết thúc ở đỉnh không có cạnh ra thì phải là `destination`. Bám sát từng dòng của đề nên đây là **oracle độc lập**.

In [5]:
def leads_brute(n: int, edges: List[List[int]], source: int, destination: int) -> bool:
    graph = [[] for _ in range(n)]
    for a, b in edges:
        graph[a].append(b)
    found = False
    stack = [(source, 0)]                              # (đỉnh cuối của đường đi, số cạnh đã đi)
    while stack:
        u, length = stack.pop()
        if length >= n:                                # n cạnh = n + 1 đỉnh: có đỉnh lặp, tức có chu trình: vô hạn đường đi
            return False
        if u == destination:
            found = True
        if not graph[u] and u != destination:          # đường tới ngõ cụt không phải đích
            return False
        for v in graph[u]:
            stack.append((v, length + 1))
    return found

for n, edges, s, d, expected in examples:
    got = leads_brute(n, edges, s, d)
    print(f"brute = {got}", "OK" if got == expected else "SAI")
    assert got == expected

brute = False OK
brute = False OK
brute = True OK


**Độ phức tạp và nút thắt.** Số đường đi có thể **tăng theo hàm mũ** (ví dụ đồ thị nhiều tầng, mỗi tầng hai đỉnh, hai cạnh tới tầng sau cho `2^k` đường). Đo thử:

In [6]:
def layered(k):
    """k tầng, mỗi tầng 2 đỉnh nối đầy đủ sang tầng kế; tầng cuối nối vào đích. Số đường đi từ nguồn = 2^k."""
    edges = [[0, 1], [0, 2]]
    node = 1
    for _ in range(k - 1):
        for a in (node, node + 1):
            for b in (node + 2, node + 3):
                edges.append([a, b])
        node += 2
    dest = node + 2
    edges.append([node, dest])
    edges.append([node + 1, dest])
    return dest + 1, edges, 0, dest

for k in (8, 12, 16, 18):
    n, edges, s, d = layered(k)
    t0 = time.perf_counter()
    res = leads_brute(n, edges, s, d)
    dt = time.perf_counter() - t0
    print(f"k={k:2} tầng, n={n:2} đỉnh: kết quả {res}, {dt * 1000:9.1f} ms")

k= 8 tầng, n=18 đỉnh: kết quả True,       0.1 ms
k=12 tầng, n=26 đỉnh: kết quả True,       1.4 ms
k=16 tầng, n=34 đỉnh: kết quả True,      24.7 ms
k=18 tầng, n=38 đỉnh: kết quả True,      96.6 ms


Nút thắt: ta liệt kê lại cùng một đỉnh qua nhiều đường khác nhau. Chỉ cần quyết định **một lần cho mỗi đỉnh**: "mọi đường từ đỉnh này có dẫn tới đích không".

## 6. Ý tưởng thuật toán / bước đột phá

**Quyết định theo đỉnh.** Gọi `good(u)` = "mọi đường đi từ `u` đều hữu hạn và kết thúc tại `destination`". Khi đó:

- Nếu `u` không có cạnh ra: `good(u) = (u == destination)`.
- Ngược lại: `good(u)` đúng khi và chỉ khi **mọi** hậu duệ `v` của `u` có `good(v)` đúng **và** không có chu trình đi qua `u`.

Đáp án là `good(source)`. Mỗi đỉnh chỉ cần tính một lần (ghi nhớ): DFS với **ba trạng thái**; gặp trạng thái `1` là chu trình; gặp trạng thái `2` là đã đúng (nếu đã sai ta đã dừng và trả về `False` ngay).

**Cách khác (sắp xếp tô-pô).** Lấy tập `R` các đỉnh đạt được từ `source`; chạy thuật toán Kahn trên `R`: nếu không xử lý hết `R` thì có chu trình; nếu xử lý hết thì kiểm tra mọi ngõ cụt trong `R` bằng `destination`.

| Cách | Ý tưởng |
|---|---|
| 2 | DFS đệ quy với ba trạng thái |
| 3 | DFS lặp với ngăn xếp tự quản lý (nộp) |
| 4 | Kahn trên các đỉnh đạt được từ `source` |

## 7. Các cách tiếp theo

### 7.1. Cách 2 — DFS đệ quy với ba trạng thái

**Giải thích từng đoạn.**

- `state[u]`: `0` chưa thăm, `1` đang trên đường đi, `2` đã xong và đúng.
- `dfs(u)`: nếu `state[u] == 1` ⇒ chu trình ⇒ `False`; nếu `state[u] == 2` ⇒ `True`.
- Nếu `u` không có cạnh ra: trả về `u == destination`.
- Ngược lại đặt `state[u] = 1`, gọi `dfs(v)` cho mọi hậu duệ (sai thì trả về `False`), rồi đặt `state[u] = 2`.

In [7]:
def leads_dfs_recursive(n: int, edges: List[List[int]], source: int, destination: int) -> bool:
    graph = [[] for _ in range(n)]
    for a, b in edges:
        graph[a].append(b)
    state = [0] * n

    def dfs(u: int) -> bool:
        if state[u] == 1:                              # đang trên đường đi hiện tại: chu trình
            return False
        if state[u] == 2:                              # đã xong, đúng
            return True
        if not graph[u]:                               # ngõ cụt phải là đích
            return u == destination
        state[u] = 1
        for v in graph[u]:
            if not dfs(v):
                return False
        state[u] = 2
        return True

    return dfs(source)

for n, edges, s, d, expected in examples:
    got = leads_dfs_recursive(n, edges, s, d)
    print(f"DFS đệ quy = {got}", "OK" if got == expected else "SAI")
    assert got == expected

DFS đệ quy = False OK
DFS đệ quy = False OK
DFS đệ quy = True OK


**Code nộp LeetCode (Cách 2):** đúng nhưng cần nâng giới hạn đệ quy cho đồ thị dài.

In [8]:
class Solution:
    def leadsToDestination(self, n: int, edges: List[List[int]], source: int, destination: int) -> bool:
        import sys
        sys.setrecursionlimit(max(sys.getrecursionlimit(), 3 * n + 100))     # đồ thị xích dài cần đệ quy sâu
        graph = [[] for _ in range(n)]
        for a, b in edges:
            graph[a].append(b)
        state = [0] * n                                # 0 chưa thăm, 1 đang trên đường đi, 2 đã xong

        def dfs(u: int) -> bool:
            if state[u] == 1:
                return False                           # chu trình
            if state[u] == 2:
                return True
            if not graph[u]:
                return u == destination                # ngõ cụt phải là đích
            state[u] = 1
            for v in graph[u]:
                if not dfs(v):
                    return False
            state[u] = 2
            return True

        return dfs(source)

sol_rec = lambda n, e, s, d: Solution().leadsToDestination(n, e, s, d)
for n, edges, s, d, expected in examples:
    assert sol_rec(n, edges, s, d) == expected
print("Cách 2 qua ví dụ")

Cách 2 qua ví dụ


**Độ phức tạp (Cách 2):** thời gian `O(n + m)`, bộ nhớ `O(n + m)` (đồ thị) và `O(n)` cho ngăn xếp đệ quy.

---

### 7.2. Cách 3 — DFS lặp với ngăn xếp tự quản lý (cách nộp chính)

**Giải thích từng đoạn.**

- `stack` chứa các đỉnh trên đường đi hiện tại; `ptr[u]` là chỉ số cạnh tiếp theo cần xét của `u`.
- Lấy `u = stack[-1]`:
  - nếu `u` là ngõ cụt: phải bằng `destination` (sai thì `False`), đánh dấu xong và bỏ ra khỏi ngăn xếp;
  - nếu còn cạnh chưa xét: lấy `v`; `state[v] == 1` ⇒ chu trình ⇒ `False`; `state[v] == 0` ⇒ vào `v` (đặt `1`, đẩy lên ngăn xếp); `state[v] == 2` ⇒ bỏ qua;
  - nếu đã xét hết cạnh: đặt `state[u] = 2` và bỏ ra.
- Nếu ngăn xếp rỗng mà không `False` thì trả `True`.

In [9]:
def leads_dfs_iterative(n: int, edges: List[List[int]], source: int, destination: int) -> bool:
    graph = [[] for _ in range(n)]
    for a, b in edges:
        graph[a].append(b)
    state = [0] * n
    ptr = [0] * n
    state[source] = 1
    stack = [source]
    while stack:
        u = stack[-1]
        if not graph[u]:                               # ngõ cụt
            if u != destination:
                return False
            state[u] = 2
            stack.pop()
        elif ptr[u] < len(graph[u]):
            v = graph[u][ptr[u]]
            ptr[u] += 1
            if state[v] == 1:                          # quay lại đỉnh trên đường đi: chu trình
                return False
            if state[v] == 0:
                state[v] = 1
                stack.append(v)
        else:                                          # đã xét hết hậu duệ
            state[u] = 2
            stack.pop()
    return True

for n, edges, s, d, expected in examples:
    got = leads_dfs_iterative(n, edges, s, d)
    print(f"DFS lặp = {got}", "OK" if got == expected else "SAI")
    assert got == expected

DFS lặp = False OK
DFS lặp = False OK
DFS lặp = True OK


**Code nộp LeetCode (Cách 3):**

In [10]:
class Solution:
    def leadsToDestination(self, n: int, edges: List[List[int]], source: int, destination: int) -> bool:
        graph = [[] for _ in range(n)]
        for a, b in edges:
            graph[a].append(b)
        state = [0] * n                                # 0 chưa thăm, 1 đang trên đường đi hiện tại, 2 đã xong
        ptr = [0] * n                                  # chỉ số cạnh kế tiếp cần xét của mỗi đỉnh
        state[source] = 1
        stack = [source]                               # ngăn xếp tự quản lý, tránh RecursionError
        while stack:
            u = stack[-1]
            if not graph[u]:                           # ngõ cụt phải là đích
                if u != destination:
                    return False
                state[u] = 2
                stack.pop()
            elif ptr[u] < len(graph[u]):
                v = graph[u][ptr[u]]
                ptr[u] += 1
                if state[v] == 1:                      # gặp lại đỉnh đang trên đường đi: chu trình
                    return False
                if state[v] == 0:
                    state[v] = 1
                    stack.append(v)
            else:                                      # mọi hậu duệ đều ổn
                state[u] = 2
                stack.pop()
        return True

sol_iter = lambda n, e, s, d: Solution().leadsToDestination(n, e, s, d)
for n, edges, s, d, expected in examples:
    assert sol_iter(n, edges, s, d) == expected
print("Cách 3 qua ví dụ")

Cách 3 qua ví dụ


**Độ phức tạp (Cách 3):** thời gian `O(n + m)`, bộ nhớ `O(n + m)`.

---

### 7.3. Cách 4 — Kahn (sắp xếp tô-pô) trên các đỉnh đạt được từ `source`

**Ý tưởng khác.** Hai việc tách riêng:

1. BFS từ `source` để lấy tập `R` các đỉnh đạt được; tính **bậc vào** của mỗi đỉnh trong `R` (chỉ tính các cạnh xuất phát từ đỉnh trong `R`; mọi cạnh đó đều trỏ vào `R`).
2. Chạy Kahn: đưa các đỉnh bậc vào `0` vào hàng đợi, lấy ra và giảm bậc vào của hậu duệ. Nếu số đỉnh xử lý `< |R|` thì có chu trình ⇒ `False`.
3. Cuối cùng mọi đỉnh trong `R` không có cạnh ra phải bằng `destination`.

In [11]:
def leads_kahn(n: int, edges: List[List[int]], source: int, destination: int) -> bool:
    graph = [[] for _ in range(n)]
    for a, b in edges:
        graph[a].append(b)
    seen = [False] * n
    seen[source] = True
    order = [source]
    head = 0
    while head < len(order):                           # BFS tìm tập R các đỉnh đạt được
        u = order[head]
        head += 1
        for v in graph[u]:
            if not seen[v]:
                seen[v] = True
                order.append(v)
    indeg = [0] * n
    for u in order:
        for v in graph[u]:
            indeg[v] += 1
    queue = deque(u for u in order if indeg[u] == 0)
    processed = 0
    while queue:
        u = queue.popleft()
        processed += 1
        for v in graph[u]:
            indeg[v] -= 1
            if indeg[v] == 0:
                queue.append(v)
    if processed != len(order):                        # còn đỉnh chưa xử lý: có chu trình
        return False
    return all(u == destination for u in order if not graph[u])

for n, edges, s, d, expected in examples:
    got = leads_kahn(n, edges, s, d)
    print(f"Kahn = {got}", "OK" if got == expected else "SAI")
    assert got == expected

Kahn = False OK
Kahn = False OK
Kahn = True OK


**Code nộp LeetCode (Cách 4):**

In [12]:
class Solution:
    def leadsToDestination(self, n: int, edges: List[List[int]], source: int, destination: int) -> bool:
        from collections import deque
        graph = [[] for _ in range(n)]
        for a, b in edges:
            graph[a].append(b)
        seen = [False] * n                             # tập R: các đỉnh đạt được từ source
        seen[source] = True
        order = [source]
        for u in order:                                # duyệt trong khi danh sách còn dài ra (BFS)
            for v in graph[u]:
                if not seen[v]:
                    seen[v] = True
                    order.append(v)
        indeg = [0] * n
        for u in order:
            for v in graph[u]:
                indeg[v] += 1
        queue = deque(u for u in order if indeg[u] == 0)
        processed = 0
        while queue:                                   # thuật toán Kahn
            u = queue.popleft()
            processed += 1
            for v in graph[u]:
                indeg[v] -= 1
                if indeg[v] == 0:
                    queue.append(v)
        if processed != len(order):                    # không sắp xếp tô-pô hết: có chu trình đạt được
            return False
        return all(u == destination for u in order if not graph[u])   # mọi ngõ cụt đạt được phải là đích

sol_kahn = lambda n, e, s, d: Solution().leadsToDestination(n, e, s, d)
for n, edges, s, d, expected in examples:
    assert sol_kahn(n, edges, s, d) == expected
print("Cách 4 qua ví dụ")

Cách 4 qua ví dụ


**Độ phức tạp (Cách 4):** thời gian `O(n + m)`, bộ nhớ `O(n + m)`.

## 8. TRACE từng bước

### 8.1. DFS lặp cho ví dụ 3 (có đỉnh kim cương)

In ngăn xếp, trạng thái và quyết định ở mỗi bước.

In [13]:
def trace_iter(n, edges, source, destination):
    graph = [[] for _ in range(n)]
    for a, b in edges:
        graph[a].append(b)
    state, ptr = [0] * n, [0] * n
    state[source] = 1
    stack = [source]
    step = 0
    while stack:
        step += 1
        u = stack[-1]
        if not graph[u]:
            if u != destination:
                print(f"{step:2}. ngõ cụt {u} khác đích {destination} -> False")
                return False
            state[u] = 2
            stack.pop()
            print(f"{step:2}. ngõ cụt {u} = đích -> xong; stack = {stack}; state = {state}")
        elif ptr[u] < len(graph[u]):
            v = graph[u][ptr[u]]
            ptr[u] += 1
            if state[v] == 1:
                print(f"{step:2}. cạnh {u} -> {v}: {v} đang trên đường đi -> chu trình -> False")
                return False
            if state[v] == 0:
                state[v] = 1
                stack.append(v)
                print(f"{step:2}. cạnh {u} -> {v}: vào {v}; stack = {stack}; state = {state}")
            else:
                print(f"{step:2}. cạnh {u} -> {v}: {v} đã xong (trạng thái 2) -> bỏ qua")
        else:
            state[u] = 2
            stack.pop()
            print(f"{step:2}. {u} hết cạnh -> xong; stack = {stack}; state = {state}")
    print("kết quả: True")
    return True

assert trace_iter(4, [[0, 1], [0, 2], [1, 3], [2, 3]], 0, 3) is True

 1. cạnh 0 -> 1: vào 1; stack = [0, 1]; state = [1, 1, 0, 0]
 2. cạnh 1 -> 3: vào 3; stack = [0, 1, 3]; state = [1, 1, 0, 1]
 3. ngõ cụt 3 = đích -> xong; stack = [0, 1]; state = [1, 1, 0, 2]
 4. 1 hết cạnh -> xong; stack = [0]; state = [1, 2, 0, 2]
 5. cạnh 0 -> 2: vào 2; stack = [0, 2]; state = [1, 2, 1, 2]
 6. cạnh 2 -> 3: 3 đã xong (trạng thái 2) -> bỏ qua
 7. 2 hết cạnh -> xong; stack = [0]; state = [1, 2, 2, 2]
 8. 0 hết cạnh -> xong; stack = []; state = [2, 2, 2, 2]
kết quả: True


### 8.2. DFS lặp cho ví dụ 2 (có chu trình)

Quan sát lúc phát hiện chu trình `1 -> 2 -> 1`.

In [14]:
assert trace_iter(4, [[0, 1], [0, 3], [1, 2], [2, 1]], 0, 3) is False
print()
assert trace_iter(3, [[0, 1], [0, 2]], 0, 2) is False

 1. cạnh 0 -> 1: vào 1; stack = [0, 1]; state = [1, 1, 0, 0]
 2. cạnh 1 -> 2: vào 2; stack = [0, 1, 2]; state = [1, 1, 1, 0]
 3. cạnh 2 -> 1: 1 đang trên đường đi -> chu trình -> False

 1. cạnh 0 -> 1: vào 1; stack = [0, 1]; state = [1, 1, 0]
 2. ngõ cụt 1 khác đích 2 -> False


### 8.3. Kahn cho ví dụ 2

In tập `R`, bậc vào, và số đỉnh xử lý được (chu trình làm Kahn "kẹt").

In [15]:
n, edges, source, destination = 4, [[0, 1], [0, 3], [1, 2], [2, 1]], 0, 3
graph = [[] for _ in range(n)]
for a, b in edges:
    graph[a].append(b)
R, seen = [source], {source}
for u in R:
    for v in graph[u]:
        if v not in seen:
            seen.add(v)
            R.append(v)
indeg = {u: 0 for u in R}
for u in R:
    for v in graph[u]:
        indeg[v] += 1
print("R =", R, "| bậc vào =", indeg)
queue, processed = deque(u for u in R if indeg[u] == 0), []
while queue:
    u = queue.popleft()
    processed.append(u)
    for v in graph[u]:
        indeg[v] -= 1
        if indeg[v] == 0:
            queue.append(v)
print("xử lý được:", processed, "| |R| =", len(R), "-> có chu trình" if len(processed) != len(R) else "-> không chu trình")
assert len(processed) < len(R)

R = [0, 1, 3, 2] | bậc vào = {0: 0, 1: 2, 3: 1, 2: 1}
xử lý được: [0, 3] | |R| = 4 -> có chu trình


## 9. Code nộp LeetCode

Ta chọn **Cách 3** (DFS lặp ba trạng thái): `O(n + m)`, không lo `RecursionError`.

```python
class Solution:
    def leadsToDestination(self, n: int, edges: List[List[int]], source: int, destination: int) -> bool:
        graph = [[] for _ in range(n)]
        for a, b in edges:
            graph[a].append(b)
        state = [0] * n                                # 0 chưa thăm, 1 đang trên đường đi hiện tại, 2 đã xong
        ptr = [0] * n                                  # chỉ số cạnh kế tiếp cần xét của mỗi đỉnh
        state[source] = 1
        stack = [source]                               # ngăn xếp tự quản lý, tránh RecursionError
        while stack:
            u = stack[-1]
            if not graph[u]:                           # ngõ cụt phải là đích
                if u != destination:
                    return False
                state[u] = 2
                stack.pop()
            elif ptr[u] < len(graph[u]):
                v = graph[u][ptr[u]]
                ptr[u] += 1
                if state[v] == 1:                      # gặp lại đỉnh đang trên đường đi: chu trình
                    return False
                if state[v] == 0:
                    state[v] = 1
                    stack.append(v)
            else:                                      # mọi hậu duệ đều ổn
                state[u] = 2
                stack.pop()
        return True
```

In [16]:
class Solution:
    def leadsToDestination(self, n: int, edges: List[List[int]], source: int, destination: int) -> bool:
        graph = [[] for _ in range(n)]
        for a, b in edges:
            graph[a].append(b)
        state = [0] * n                                # 0 chưa thăm, 1 đang trên đường đi hiện tại, 2 đã xong
        ptr = [0] * n                                  # chỉ số cạnh kế tiếp cần xét của mỗi đỉnh
        state[source] = 1
        stack = [source]                               # ngăn xếp tự quản lý, tránh RecursionError
        while stack:
            u = stack[-1]
            if not graph[u]:                           # ngõ cụt phải là đích
                if u != destination:
                    return False
                state[u] = 2
                stack.pop()
            elif ptr[u] < len(graph[u]):
                v = graph[u][ptr[u]]
                ptr[u] += 1
                if state[v] == 1:                      # gặp lại đỉnh đang trên đường đi: chu trình
                    return False
                if state[v] == 0:
                    state[v] = 1
                    stack.append(v)
            else:                                      # mọi hậu duệ đều ổn
                state[u] = 2
                stack.pop()
        return True

sol = lambda n, e, s, d: Solution().leadsToDestination(n, e, s, d)
for n, edges, s, d, expected in examples:
    got = sol(n, edges, s, d)
    print(got, "OK" if got == expected else "SAI")
    assert got == expected

False OK
False OK
True OK


## 10. Vì sao đúng

**Dạng gọn của đề.** Gọi `R` là tập đỉnh đạt được từ `source`.

- Nếu có chu trình đạt được từ `source` thì có vô hạn đường đi (đi vòng tùy ý) nên điều kiện (3) sai ⇒ `False`.
- Nếu không có chu trình trong `R`, mọi đường đi là hữu hạn; mỗi đường kéo dài được cho đến khi tới một đỉnh không có cạnh ra (vì không thể đi mãi). Do đó điều kiện (2) tương đương: mọi ngõ cụt trong `R` bằng `destination`. Vì `R` hữu hạn và không có chu trình nên `R` có ít nhất một ngõ cụt, nó phải bằng `destination` ⇒ điều kiện (1) tự động đúng.
- Nếu `destination` có cạnh ra trong `R` thì đường đi tới `destination` còn kéo dài tới một ngõ cụt khác `destination` (đồ thị không chu trình) ⇒ vi phạm điều kiện (2) ⇒ `False`; thuật toán xử lý trường hợp này tự nhiên vì kiểm tra ngõ cụt và chu trình.

Vậy kết quả đúng là: **không có chu trình đạt được từ `source` VÀ mọi ngõ cụt đạt được đều bằng `destination`**.

**DFS ba trạng thái.** Trạng thái `1` là các đỉnh trên đường đi DFS hiện tại; một cạnh tới đỉnh trạng thái `1` đóng một chu trình, và ngược lại mọi chu trình đạt được đều bị DFS phát hiện (đỉnh đầu tiên của chu trình được DFS thăm sẽ còn ở trạng thái `1` khi DFS đi hết chu trình). Cạnh tới đỉnh trạng thái `2` an toàn vì đỉnh đó và mọi đỉnh sau nó đã được kiểm tra đúng. Mỗi đỉnh vào ngăn xếp tối đa một lần và mỗi cạnh được xét một lần ⇒ `O(n + m)`.

**Kahn.** Nếu `R` không có chu trình thì có thể sắp xếp tô-pô nên Kahn xử lý hết `R`; nếu có chu trình thì các đỉnh trên chu trình (và sau nó) không bao giờ có bậc vào về `0`, nên số đỉnh xử lý `< |R|`.

**Kiểm chứng bằng máy.** Oracle (Cách 1) liệt kê mọi đường đi có giới hạn độ dài theo đúng định nghĩa. Ta đối chiếu mọi cách với oracle trên **mọi** đồ thị có `n <= 3` đỉnh (kể cả khuyên và với mọi cặp `source`, `destination`) và nhiều đồ thị ngẫu nhiên `n <= 7` (có cạnh song song).

In [17]:
impls = {"DFS đệ quy": leads_dfs_recursive, "DFS lặp": leads_dfs_iterative, "Kahn": leads_kahn,
         "nộp (Cách 3)": sol, "Cách 2 (nộp)": sol_rec, "Cách 4 (nộp)": sol_kahn}
total = 0
for n in (1, 2, 3):
    pairs = [(a, b) for a in range(n) for b in range(n)]               # mọi cạnh có thể, kể cả khuyên
    for mask in range(1 << len(pairs)):
        edges = [list(pairs[k]) for k in range(len(pairs)) if mask >> k & 1]
        for s in range(n):
            for d in range(n):
                ref = leads_brute(n, edges, s, d)
                for name, fn in impls.items():
                    assert fn(n, edges, s, d) == ref, (name, n, edges, s, d)
                total += 1
print("đã đối chiếu", total, "đồ thị (n <= 3, mọi tập cạnh, mọi source, destination): mọi cách khớp oracle")

đã đối chiếu 4674 đồ thị (n <= 3, mọi tập cạnh, mọi source, destination): mọi cách khớp oracle


## 11. Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Liệt kê đường đi | mũ theo số đường đi | `O(n)` ngăn xếp (nhiều trạng thái) |
| 2. DFS đệ quy | `O(n + m)` | `O(n + m)` + độ sâu đệ quy |
| 3. DFS lặp | `O(n + m)` | `O(n + m)` |
| 4. Kahn | `O(n + m)` | `O(n + m)` |

Đo ở cỡ tối đa (`n = m = 10^4`) với các dạng: xích dài, DAG ngẫu nhiên, chu trình lớn, hình sao, nhiều tầng kim cương:

In [18]:
def chain(n):
    return n, [[i, i + 1] for i in range(n - 1)], 0, n - 1

def random_dag_to_sink(rng, n, m):
    edges = set()
    while len(edges) < m:
        a = rng.randint(0, n - 2)
        b = rng.randint(a + 1, n - 1)
        edges.add((a, b))
    edges = [list(e) for e in edges]
    has_out = {a for a, _ in edges}
    for v in range(n - 1):
        if v not in has_out:
            edges.append([v, n - 1])
    return n, edges, 0, n - 1

rng = random.Random(11)
N = 10 ** 4
cases = {
    "xích 10^4 đỉnh": chain(N),
    "DAG ngẫu nhiên về đích": random_dag_to_sink(rng, N, 9000),
    "chu trình lớn": (N, [[i, (i + 1) % N] for i in range(N)], 0, N - 1),
    "hình sao: nguồn nối 9999 ngõ cụt khác đích": (N, [[0, v] for v in range(1, N)], 0, N - 1),
    "hình sao mọi nhánh về đích": (N, [[0, v] for v in range(1, N - 1)] + [[v, N - 1] for v in range(1, N - 1)], 0, N - 1),
    "khuyên ở đích": (3, [[0, 1], [1, 2], [2, 2]], 0, 2),
}
old = sys.getrecursionlimit()
for label, (n, e, s, d) in cases.items():
    ref = sol(n, e, s, d)
    row = [f"{label:44}"]
    for name, fn in (("DFS lặp", sol_iter), ("Kahn", sol_kahn), ("DFS đệ quy", sol_rec)):
        t0 = time.perf_counter()
        got = fn(n, e, s, d)
        dt = (time.perf_counter() - t0) * 1000
        assert got == ref, (label, name)
        row.append(f"{name} {dt:7.2f} ms")
    print(" | ".join(row), "| kết quả", ref)
sys.setrecursionlimit(old)

xích 10^4 đỉnh                               | DFS lặp    3.87 ms | Kahn    3.52 ms | DFS đệ quy    3.28 ms | kết quả True


DAG ngẫu nhiên về đích                       | DFS lặp   24.76 ms | Kahn    1.30 ms | DFS đệ quy    1.34 ms | kết quả True


chu trình lớn                                | DFS lặp    3.10 ms | Kahn    2.65 ms | DFS đệ quy    2.88 ms | kết quả False
hình sao: nguồn nối 9999 ngõ cụt khác đích   | DFS lặp    0.82 ms | Kahn    0.89 ms | DFS đệ quy    0.77 ms | kết quả False


hình sao mọi nhánh về đích                   | DFS lặp    5.92 ms | Kahn    5.63 ms | DFS đệ quy    5.51 ms | kết quả True
khuyên ở đích                                | DFS lặp    0.00 ms | Kahn    0.00 ms | DFS đệ quy    0.00 ms | kết quả False


## 12. Bộ test

Gồm: ví dụ của đề, edge case có kỳ vọng viết tay (kèm lý do), phản ví dụ cho các cách sai (tìm bằng tìm kiếm so với oracle), test ngẫu nhiên có seed (có khuyên, cạnh song song) và test cỡ lớn.

In [19]:
def check_all(n, edges, source, destination, expected=None, brute=True):
    results = {name: fn(n, edges, source, destination) for name, fn in impls.items()}
    first = results["nộp (Cách 3)"]
    for name, got in results.items():
        assert got == first, (name, got, first, n, edges[:10])
    if brute and n <= 9 and len(edges) <= 14:
        assert leads_brute(n, edges, source, destination) == first, (n, edges, source, destination)
    if expected is not None:
        assert first == expected, (n, edges, source, destination, first, expected)
    return first

# 12.1 ví dụ đề
for n, e, s, d, ex in examples:
    check_all(n, e, s, d, ex)
print("12.1 ví dụ đề: OK")

# 12.2 edge case (kỳ vọng viết tay, kèm lý do)
edge = [
    (1, [], 0, 0, True),                                     # một đỉnh: source = destination, không có cạnh ra
    (2, [], 0, 1, False),                                    # không có cạnh: source là ngõ cụt khác đích
    (1, [[0, 0]], 0, 0, False),                              # khuyên ở source = destination: vô hạn đường đi
    (2, [[0, 1]], 0, 1, True),                               # một cạnh
    (2, [[0, 1], [0, 1]], 0, 1, True),                       # cạnh song song
    (3, [[0, 1], [1, 2], [2, 2]], 0, 2, False),              # khuyên ở đích
    (3, [[0, 1], [1, 2]], 0, 1, False),                      # đích có cạnh ra: đường tiếp tục tới ngõ cụt 2 khác đích
    (3, [[0, 1], [1, 2]], 0, 2, True),                       # chuỗi thẳng
    (3, [[0, 1], [1, 0], [0, 2]], 0, 2, False),              # chu trình 0 <-> 1 đạt được
    (4, [[0, 1], [0, 2], [1, 3], [2, 3]], 0, 3, True),       # kim cương (ví dụ 3)
    (4, [[0, 1], [2, 3]], 0, 1, True),                       # phần đồ thị không đạt được (2 -> 3) bị bỏ qua
    (4, [[0, 1], [2, 2]], 0, 1, True),                       # khuyên ở đỉnh không đạt được: không ảnh hưởng
    (4, [[0, 1], [2, 3]], 0, 3, False),                      # đích không đạt được từ source
    (3, [[0, 1], [0, 2], [1, 2]], 0, 2, True),               # đường tắt và đường dài cùng tới 2
    (3, [[0, 1], [0, 2], [1, 2]], 0, 1, False),              # đích 1 có cạnh ra, 2 là ngõ cụt khác đích
    (5, [[0, 1], [1, 2], [2, 3], [3, 1], [0, 4]], 0, 4, False),   # chu trình 1 -> 2 -> 3 -> 1
    (2, [[1, 0]], 0, 0, True),                               # source = destination là ngõ cụt; cạnh 1 -> 0 không đạt được từ 0
]
for n, e, s, d, ex in edge:
    check_all(n, e, s, d, ex)
print("12.2 edge case: OK", len(edge), "ca")

12.1 ví dụ đề: OK
12.2 edge case: OK 17 ca


In [20]:
# 12.3 phản ví dụ: các cách sai (tìm bằng tìm kiếm so với oracle)
def build(n, edges):
    graph = [[] for _ in range(n)]
    for a, b in edges:
        graph[a].append(b)
    return graph

def wrong_reachable_only(n, edges, source, destination):         # chỉ kiểm tra đến được destination
    graph = build(n, edges)
    seen, stack = {source}, [source]
    while stack:
        u = stack.pop()
        for v in graph[u]:
            if v not in seen:
                seen.add(v)
                stack.append(v)
    return destination in seen

def wrong_no_cycle_check(n, edges, source, destination):         # chỉ kiểm tra ngõ cụt, bỏ qua chu trình
    graph = build(n, edges)
    seen, stack = {source}, [source]
    while stack:
        u = stack.pop()
        if not graph[u] and u != destination:
            return False
        for v in graph[u]:
            if v not in seen:
                seen.add(v)
                stack.append(v)
    return destination in seen

def wrong_stop_at_destination(n, edges, source, destination):    # dừng ngay khi tới destination, không xét cạnh ra của destination
    graph = build(n, edges)
    state = [0] * n

    def dfs(u):
        if u == destination:
            return True
        if state[u] == 1:
            return False
        if state[u] == 2:
            return True
        if not graph[u]:
            return False
        state[u] = 1
        for v in graph[u]:
            if not dfs(v):
                return False
        state[u] = 2
        return True

    return dfs(source)

def wrong_single_visited(n, edges, source, destination):         # một mảng visited: gặp lại đỉnh là coi như chu trình
    graph = build(n, edges)
    visited = [False] * n

    def dfs(u):
        if visited[u]:
            return False
        visited[u] = True
        if not graph[u]:
            return u == destination
        return all(dfs(v) for v in graph[u])

    return dfs(source)

def wrong_all_nodes(n, edges, source, destination):              # kiểm tra mọi đỉnh thay vì chỉ các đỉnh đạt được từ source
    graph = build(n, edges)
    state = [0] * n

    def dfs(u):
        if state[u] == 1:
            return False
        if state[u] == 2:
            return True
        if not graph[u]:
            return u == destination
        state[u] = 1
        for v in graph[u]:
            if not dfs(v):
                return False
        state[u] = 2
        return True

    return all(dfs(u) for u in range(n))

def find_counter(wrong, tries=40000, seed=9):
    """Tìm phản ví dụ NHỎ NHẤT (ít cạnh nhất, rồi ít đỉnh nhất), cạnh không lặp."""
    rng = random.Random(seed)
    best = None
    for _ in range(tries):
        n = rng.randint(1, 4)
        m = rng.randint(0, 5)
        edges = [list(e) for e in sorted({(rng.randrange(n), rng.randrange(n)) for _ in range(m)})]
        s, d = rng.randrange(n), rng.randrange(n)
        if best is not None and (len(edges), n) >= (len(best[1]), best[0]):
            continue
        if wrong(n, edges, s, d) != leads_brute(n, edges, s, d):
            best = (n, edges, s, d)
    return best

hand_cases = {                                       # phản ví dụ chọn tay (dễ nhìn), được xác nhận lại bằng oracle
    "chỉ kiểm tra đến được destination": (wrong_reachable_only, 3, [[0, 1], [0, 2]], 0, 2),                    # ví dụ 1 của đề
    "bỏ qua chu trình": (wrong_no_cycle_check, 4, [[0, 1], [0, 3], [1, 2], [2, 1]], 0, 3),                     # ví dụ 2 của đề
    "dừng ngay khi tới destination": (wrong_stop_at_destination, 3, [[0, 1], [1, 2]], 0, 1),                    # đích 1 còn cạnh ra tới ngõ cụt 2
    "một mảng visited duy nhất": (wrong_single_visited, 4, [[0, 1], [0, 2], [1, 3], [2, 3]], 0, 3),            # ví dụ 3 của đề (kim cương)
    "kiểm tra mọi đỉnh thay vì chỉ từ source": (wrong_all_nodes, 4, [[0, 1], [2, 3]], 0, 1),                    # cạnh 2 -> 3 không đạt được từ 0
}
counters = {}
for name, (fn, n, e, s, d) in hand_cases.items():
    assert fn(n, e, s, d) != leads_brute(n, e, s, d), name
    assert find_counter(fn) is not None, name                     # tìm kiếm tự động cũng tìm ra phản ví dụ (nhỏ nhất)
    counters[name] = (n, e, s, d)
    print(f"{name:42} n={n} edges={e} source={s} destination={d}: sai -> {fn(n, e, s, d)}, đúng -> {leads_brute(n, e, s, d)}")
    check_all(n, e, s, d)
print("12.3 phản ví dụ: OK")

chỉ kiểm tra đến được destination          n=3 edges=[[0, 1], [0, 2]] source=0 destination=2: sai -> True, đúng -> False


bỏ qua chu trình                           n=4 edges=[[0, 1], [0, 3], [1, 2], [2, 1]] source=0 destination=3: sai -> True, đúng -> False


dừng ngay khi tới destination              n=3 edges=[[0, 1], [1, 2]] source=0 destination=1: sai -> True, đúng -> False


một mảng visited duy nhất                  n=4 edges=[[0, 1], [0, 2], [1, 3], [2, 3]] source=0 destination=3: sai -> False, đúng -> True
kiểm tra mọi đỉnh thay vì chỉ từ source    n=4 edges=[[0, 1], [2, 3]] source=0 destination=1: sai -> False, đúng -> True
12.3 phản ví dụ: OK


In [21]:
# 12.4 ngẫu nhiên nhỏ: có khuyên, cạnh song song, nhiều dạng mật độ
rng = random.Random(4059)
for _ in range(4000):
    n = rng.randint(1, 7)
    m = rng.randint(0, 2 * n)
    edges = [[rng.randrange(n), rng.randrange(n)] for _ in range(m)]
    check_all(n, edges, rng.randrange(n), rng.randrange(n))
print("12.4 ngẫu nhiên nhỏ (đồ thị tùy ý): OK")

# 12.5 DAG ngẫu nhiên (a < b): không chu trình, so oracle và so các cách (kết quả phụ thuộc ngõ cụt)
true_count = 0
for _ in range(4000):
    n = rng.randint(2, 8)
    edges = [[a, rng.randint(a + 1, n - 1)] for a in range(n - 1) for _ in range(rng.randint(0, 2))]
    for a in range(n - 1):                                              # nhiều lần ép mọi đỉnh trừ n - 1 có cạnh ra
        if rng.random() < 0.7 and not any(e[0] == a for e in edges):
            edges.append([a, n - 1])
    ans = check_all(n, edges, 0, n - 1)
    true_count += ans
print("12.5 DAG ngẫu nhiên: OK, số ca True =", true_count, "/ 4000")
assert 200 < true_count < 3800                                        # cả hai kết quả đều xuất hiện

# 12.6 ngẫu nhiên lớn: các cách nhanh so với nhau (không dùng oracle)
for _ in range(15):
    n = rng.randint(2000, 10 ** 4)
    m = rng.randint(n // 2, 10 ** 4)
    if rng.random() < 0.5:
        edges = [[a, rng.randint(a + 1, n - 1)] for a in rng.sample(range(n - 1), min(m, n - 1))]
        has_out = {e[0] for e in edges}
        edges += [[a, n - 1] for a in range(n - 1) if a not in has_out]       # mọi đỉnh trừ n - 1 đều có cạnh ra
    else:
        edges = [[rng.randrange(n), rng.randrange(n)] for _ in range(m)]
    results = {leads_kahn(n, edges, 0, n - 1), sol_iter(n, edges, 0, n - 1), sol_kahn(n, edges, 0, n - 1)}
    assert len(results) == 1
print("12.6 ngẫu nhiên lớn: OK")

# 12.7 tính chất: thêm khuyên ở đích khi đích đạt được thì luôn False; thêm cạnh song song không đổi kết quả; thêm khuyên ở đỉnh không đạt được không đổi kết quả
def reachable(n, edges, source):
    g = build(n, edges)
    seen, st = {source}, [source]
    while st:
        u = st.pop()
        for v in g[u]:
            if v not in seen:
                seen.add(v)
                st.append(v)
    return seen

for _ in range(400):
    n = rng.randint(2, 7)
    edges = [[a, rng.randint(a + 1, n - 1)] for a in range(n - 1) for _ in range(rng.randint(1, 2))]
    ans = sol(n, edges, 0, n - 1)
    R = reachable(n, edges, 0)
    with_loop = edges + [[n - 1, n - 1]]
    assert sol(n, with_loop, 0, n - 1) == leads_brute(n, with_loop, 0, n - 1)
    if n - 1 in R:
        assert sol(n, with_loop, 0, n - 1) is False                          # khuyên ở đích đạt được: vô hạn đường đi
    assert sol(n, edges + [edges[0]], 0, n - 1) == ans                        # thêm cạnh song song: không đổi
    outside = [u for u in range(n) if u not in R]
    if outside:
        u = rng.choice(outside)
        assert sol(n, edges + [[u, u]], 0, n - 1) == ans                      # khuyên ở đỉnh không đạt được: không đổi
print("12.7 tính chất: OK")

12.4 ngẫu nhiên nhỏ (đồ thị tùy ý): OK


12.5 DAG ngẫu nhiên: OK, số ca True = 3285 / 4000


12.6 ngẫu nhiên lớn: OK
12.7 tính chất: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Chỉ kiểm tra đến được `destination`

Đề đòi **mọi** đường đều dẫn tới `destination`, không chỉ một đường.

In [22]:
n, e, s, d = counters["chỉ kiểm tra đến được destination"]
print(f"n={n} edges={e} source={s} destination={d} | sai ->", wrong_reachable_only(n, e, s, d), "| đúng ->", sol(n, e, s, d))
assert wrong_reachable_only(n, e, s, d) != sol(n, e, s, d)

n=3 edges=[[0, 1], [0, 2]] source=0 destination=2 | sai -> True | đúng -> False


Vì sao sai: có thể có ngõ cụt khác `destination` hoặc chu trình đạt được từ `source`. **Đúng:** kiểm tra mọi ngõ cụt đạt được và phát hiện chu trình.

### Lỗi 2 — Bỏ qua chu trình

Ví dụ 2 của đề: kết thúc ở `3` hoặc lặp vô hạn quanh `1, 2`.

In [23]:
n, e, s, d = counters["bỏ qua chu trình"]
print(f"n={n} edges={e} source={s} destination={d} | sai ->", wrong_no_cycle_check(n, e, s, d), "| đúng ->", sol(n, e, s, d))
assert wrong_no_cycle_check(n, e, s, d) != sol(n, e, s, d)
print("ví dụ 2 của đề:", wrong_no_cycle_check(4, [[0, 1], [0, 3], [1, 2], [2, 1]], 0, 3), "(đúng là False)")

n=4 edges=[[0, 1], [0, 3], [1, 2], [2, 1]] source=0 destination=3 | sai -> True | đúng -> False
ví dụ 2 của đề: True (đúng là False)


Vì sao sai: một đường đi vô hạn không kết thúc ở ngõ cụt nào nên kiểm tra ngõ cụt không thấy. **Đúng:** phát hiện chu trình bằng trạng thái `1` (đang trên đường đi).

### Lỗi 3 — Dừng ngay khi tới `destination`

`destination` mà còn cạnh đi ra thì đường đi tiếp tục và sẽ kết thúc ở một ngõ cụt khác.

In [24]:
n, e, s, d = counters["dừng ngay khi tới destination"]
print(f"n={n} edges={e} source={s} destination={d} | sai ->", wrong_stop_at_destination(n, e, s, d), "| đúng ->", sol(n, e, s, d))
assert wrong_stop_at_destination(n, e, s, d) != sol(n, e, s, d)

n=3 edges=[[0, 1], [1, 2]] source=0 destination=1 | sai -> True | đúng -> False


Vì sao sai: "đường đi kết thúc tại `destination`" nghĩa là **không thể đi tiếp** từ `destination`. **Đúng:** coi `destination` như mọi đỉnh khác; ngõ cụt phải là `destination` và `destination` phải là ngõ cụt (tự nhiên suy ra từ việc kiểm tra ngõ cụt).

### Lỗi 4 — Dùng một mảng `visited` duy nhất

Đường đi "kim cương" (hai đường khác nhau tới cùng một đỉnh) không phải chu trình, nhưng một mảng `visited` coi lần gặp thứ hai là chu trình.

In [25]:
n, e, s, d = counters["một mảng visited duy nhất"]
print(f"n={n} edges={e} source={s} destination={d} | sai ->", wrong_single_visited(n, e, s, d), "| đúng ->", sol(n, e, s, d))
assert wrong_single_visited(n, e, s, d) != sol(n, e, s, d)
print("ví dụ 3 của đề:", wrong_single_visited(4, [[0, 1], [0, 2], [1, 3], [2, 3]], 0, 3), "(đúng là True)")

n=4 edges=[[0, 1], [0, 2], [1, 3], [2, 3]] source=0 destination=3 | sai -> False | đúng -> True
ví dụ 3 của đề: False (đúng là True)


Vì sao sai: cần phân biệt "đang trên đường đi hiện tại" (trạng thái `1`, chu trình) với "đã xong" (trạng thái `2`, an toàn). **Đúng:** ba trạng thái.

### Lỗi 5 — Kiểm tra mọi đỉnh thay vì chỉ các đỉnh đạt được từ `source`

In [26]:
n, e, s, d = counters["kiểm tra mọi đỉnh thay vì chỉ từ source"]
print(f"n={n} edges={e} source={s} destination={d} | sai ->", wrong_all_nodes(n, e, s, d), "| đúng ->", sol(n, e, s, d))
assert wrong_all_nodes(n, e, s, d) != sol(n, e, s, d)

n=4 edges=[[0, 1], [2, 3]] source=0 destination=1 | sai -> False | đúng -> True


Vì sao sai: chu trình hoặc ngõ cụt ở phần **không đạt được** từ `source` không ảnh hưởng. **Đúng:** chỉ DFS từ `source`.

### Lỗi 6 — Đệ quy quá sâu

Đồ thị xích dài làm DFS đệ quy vượt giới hạn mặc định.

In [27]:
def deep_recursive(n, edges, source, destination):
    graph = build(n, edges)
    state = [0] * n

    def dfs(u):
        if state[u] == 1:
            return False
        if state[u] == 2:
            return True
        if not graph[u]:
            return u == destination
        state[u] = 1
        for v in graph[u]:
            if not dfs(v):
                return False
        state[u] = 2
        return True

    return dfs(source)

N = 5000
n, e, s, d = chain(N)
old = sys.getrecursionlimit()
sys.setrecursionlimit(1000)
try:
    deep_recursive(n, e, s, d)
    print("không lỗi (không mong đợi)")
except RecursionError as ex:
    print("RecursionError:", ex)
finally:
    sys.setrecursionlimit(old)
print("bản lặp trên cùng đồ thị:", sol(n, e, s, d))
assert sol(n, e, s, d) is True

RecursionError: maximum recursion depth exceeded
bản lặp trên cùng đồ thị: True


Vì sao sai: mỗi đỉnh trên xích là một khung đệ quy, vượt giới hạn mặc định `1000`. **Đúng:** DFS lặp với ngăn xếp tự quản lý, hoặc nâng `sys.setrecursionlimit` (cẩn thận).

## 14. Mẹo & nhận dạng pattern

- **Dấu hiệu:** "mọi đường đi từ nguồn đều thỏa điều kiện / không có chu trình đạt được" ⇒ **DFS ba trạng thái** hoặc **Kahn** trên phần đạt được.
- Ba trạng thái: `0` chưa thăm, `1` đang trên đường đi (gặp lại = chu trình), `2` đã xong (ghi nhớ).
- Chỉ xét các đỉnh **đạt được từ nguồn**.
- Với đồ thị lớn trong Python: DFS **lặp** (ngăn xếp tự quản lý), tránh `RecursionError`.
- Khuyên (`u -> u`) là chu trình độ dài `1`; cạnh song song không ảnh hưởng kết quả nhưng ảnh hưởng bậc vào khi dùng Kahn.
- Cùng họ: Course Schedule (207), Find Eventual Safe States (802), All Paths From Source to Target (797), Cycle Detection.
- Test nên có: `source = destination`, khuyên ở đích, chu trình không đạt được, đích có cạnh ra, kim cương, xích `10^4`.

## 15. Biến thể

1. **Find Eventual Safe States (802):** đỉnh "an toàn" nếu mọi đường đi từ nó đều dẫn đến ngõ cụt (hữu hạn). Dùng đồ thị đảo + Kahn hoặc DFS ba trạng thái.
2. **Đếm số đường đi** trong DAG (bài tập 5).
3. **Có nhiều đích** (tập `destinations`): thay `u == destination` bằng `u in destinations`.
4. Bài liên quan: 207, 797, 802.

Kiểm biến thể 1 (an toàn) bằng oracle liệt kê đường đi giới hạn độ dài:

In [28]:
def safe_nodes(n: int, graph: List[List[int]]) -> List[int]:
    """Đồ thị đảo + Kahn: đỉnh an toàn nếu mọi đường đi từ nó đều dẫn tới ngõ cụt."""
    rev = [[] for _ in range(n)]
    outdeg = [len(graph[u]) for u in range(n)]
    for u in range(n):
        for v in graph[u]:
            rev[v].append(u)
    queue = deque(u for u in range(n) if outdeg[u] == 0)
    safe = [False] * n
    while queue:
        u = queue.popleft()
        safe[u] = True
        for p in rev[u]:
            outdeg[p] -= 1
            if outdeg[p] == 0:
                queue.append(p)
    return [u for u in range(n) if safe[u]]

def safe_nodes_brute(n, graph):
    res = []
    for start in range(n):
        ok = True
        stack = [(start, 0)]
        while stack:
            u, length = stack.pop()
            if length >= n:                              # đường đi n cạnh: đi vào chu trình
                ok = False
                break
            for v in graph[u]:
                stack.append((v, length + 1))
        if ok:
            res.append(start)
    return res

rng = random.Random(15)
for _ in range(1500):
    n = rng.randint(1, 7)
    graph = [[rng.randrange(n) for _ in range(rng.randint(0, 2))] for _ in range(n)]
    assert safe_nodes(n, graph) == safe_nodes_brute(n, graph), graph
assert safe_nodes(7, [[1, 2], [2, 3], [5], [0], [5], [], []]) == [2, 4, 5, 6]
print("biến thể Find Eventual Safe States: Kahn trên đồ thị đảo khớp oracle (và ví dụ của bài 802)")

biến thể Find Eventual Safe States: Kahn trên đồ thị đảo khớp oracle (và ví dụ của bài 802)


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Liệt kê đường đi | Theo đúng định nghĩa | mũ | `O(n)` | Oracle cho đồ thị rất nhỏ |
| 2. DFS đệ quy | Ba trạng thái | `O(n + m)` | `O(n + m)` + đệ quy | Ngắn gọn, đồ thị không sâu |
| 3. DFS lặp | Ba trạng thái, ngăn xếp tự quản lý | `O(n + m)` | `O(n + m)` | **Cách nộp chính** (an toàn với `n = 10^4`) |
| 4. Kahn | Sắp xếp tô-pô trên phần đạt được | `O(n + m)` | `O(n + m)` | Khi cần thứ tự tô-pô; không đệ quy |

## 17. Bài tập vận dụng

Làm tuần tự từ dễ đến khó. Các cell có chữ `TODO` **không được gọi** khi chạy notebook; bạn tự điền rồi bỏ dấu `#` ở các dòng `assert` để kiểm tra. Đáp án nằm ở mục 19.

### Bài 1 — Đoán output (dễ)

Không chạy code, hãy đoán rồi chạy để kiểm tra:

```python
n, edges = 4, [[0, 1], [0, 2], [2, 3], [1, 3], [0, 1]]
graph = [[] for _ in range(n)]
for a, b in edges:
    graph[a].append(b)
print(graph)
print([u for u in range(n) if not graph[u]])
print(Solution().leadsToDestination(n, edges, 0, 3))
```

In [29]:
# Bài 1: ghi dự đoán của bạn
my_guess_1 = None      # danh sách kề (chú ý cạnh song song)
my_guess_2 = None      # danh sách ngõ cụt
my_guess_3 = None      # kết quả leadsToDestination

### Bài 2 — Tính tay (dễ-vừa)

`n = 5`, `edges = [[0,1],[1,2],[2,3],[3,1],[0,4]]`, `source = 0`, `destination = 4`. Ghi kết quả và nêu lý do (có chu trình nào đạt được không?).

In [30]:
# Bài 2: điền kết quả tính tay
hand_answer = None

### Bài 3 — Điền chỗ trống (vừa)

Hoàn thành DFS lặp ba trạng thái.

In [31]:
def exercise_3(n, edges, source, destination):
    graph = [[] for _ in range(n)]
    for a, b in edges:
        graph[a].append(b)
    state, ptr = [0] * n, [0] * n
    state[source] = 1
    stack = [source]
    while stack:
        u = stack[-1]
        if not graph[u]:
            # TODO 1: nếu u khác destination thì trả về False; rồi đánh dấu xong và pop
            pass
        elif ptr[u] < len(graph[u]):
            v = graph[u][ptr[u]]
            ptr[u] += 1
            # TODO 2: nếu state[v] == 1 trả về False (chu trình)
            # TODO 3: nếu state[v] == 0 thì đặt state[v] = 1 và đẩy v lên stack
            pass
        else:
            # TODO 4: đánh dấu xong và pop
            pass
    return True


# for n, e, s, d, ex in examples:
#     assert exercise_3(n, e, s, d) == ex

### Bài 4 — Tự viết lại (vừa-khó)

Không nhìn lại Cách 4, tự viết `exercise_4(n, edges, source, destination)` bằng **Kahn** trên các đỉnh đạt được từ `source`.

In [32]:
def exercise_4(n, edges, source, destination):
    # TODO
    pass


# for n, e, s, d, ex in examples:
#     assert exercise_4(n, e, s, d) == ex

### Bài 5 — Mini challenge: đếm số đường đi trong DAG (khó)

Cho đồ thị có hướng **không có chu trình** (đảm bảo mọi cạnh `a -> b` có `a < b`), có thể có **cạnh song song** (mỗi cạnh tính là một cách đi khác nhau). Viết `exercise_5(n, edges, source, destination)` trả về **số đường đi** từ `source` đến `destination`.

```python
exercise_5(4, [[0,1],[0,2],[1,3],[2,3]], 0, 3)       -> 2
exercise_5(3, [[0,1],[0,1],[1,2]], 0, 2)             -> 2
exercise_5(3, [[0,1]], 0, 2)                         -> 0
exercise_5(1, [], 0, 0)                              -> 1
```
Gợi ý: quy hoạch động theo thứ tự đỉnh tăng dần (hoặc DFS có nhớ).

In [33]:
def exercise_5(n, edges, source, destination):
    # TODO
    pass


def paths_oracle(n, edges, source, destination):
    graph = [[] for _ in range(n)]
    for a, b in edges:
        graph[a].append(b)
    count, stack = 0, [source]
    while stack:
        u = stack.pop()
        if u == destination:
            count += 1
        stack.extend(graph[u])
    return count


# assert exercise_5(4, [[0, 1], [0, 2], [1, 3], [2, 3]], 0, 3) == 2
# assert exercise_5(3, [[0, 1], [0, 1], [1, 2]], 0, 2) == 2
# assert exercise_5(3, [[0, 1]], 0, 2) == 0
# assert exercise_5(1, [], 0, 0) == 1

## 18. Gợi ý 3 mức

**Bài 1**
- Mức 1: `graph[a].append(b)` cho mỗi cạnh, kể cả cạnh lặp.
- Mức 2: đỉnh `0` có các cạnh tới `1, 2, 1` (theo thứ tự trong `edges`).
- Mức 3: `[[1, 2, 1], [3], [3], []]`; ngõ cụt `[3]`; kết quả `True` (không có chu trình, mọi đường kết thúc ở `3`).

**Bài 2**
- Mức 1: tìm chu trình đạt được từ `0`.
- Mức 2: `0 -> 1 -> 2 -> 3 -> 1` quay lại `1`: chu trình `1 -> 2 -> 3 -> 1`.
- Mức 3: `False`.

**Bài 3**
- Mức 1: ngõ cụt: `if u != destination: return False`, rồi `state[u] = 2; stack.pop()`.
- Mức 2: `if state[v] == 1: return False`.
- Mức 3: `if state[v] == 0: state[v] = 1; stack.append(v)`; khi hết cạnh: `state[u] = 2; stack.pop()`.

**Bài 4**
- Mức 1: BFS tìm tập `R`, tính `indeg` chỉ trên các cạnh xuất phát từ `R`.
- Mức 2: Kahn: nếu số đỉnh xử lý khác `len(R)` thì có chu trình.
- Mức 3: cuối cùng `all(u == destination for u in R if not graph[u])`.

**Bài 5**
- Mức 1: `ways[source] = 1`.
- Mức 2: duyệt `a` từ nhỏ đến lớn; với mỗi cạnh `a -> b`: `ways[b] += ways[a]`.
- Mức 3: trả `ways[destination]`.

## 19. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này. Mỗi cell dưới đây đều tự kiểm tra bằng `assert`.

### Đáp án Bài 1

In [34]:
n, edges = 4, [[0, 1], [0, 2], [2, 3], [1, 3], [0, 1]]
graph = [[] for _ in range(n)]
for a, b in edges:
    graph[a].append(b)
r1, r2, r3 = graph, [u for u in range(n) if not graph[u]], Solution().leadsToDestination(n, edges, 0, 3)
print(r1, r2, r3)
assert r1 == [[1, 2, 1], [3], [3], []]
assert r2 == [3]
assert r3 is True

[[1, 2, 1], [3], [3], []] [3] True


### Đáp án Bài 2

Từ `0` đạt được `1, 2, 3, 4`. Có chu trình `1 -> 2 -> 3 -> 1` đạt được, nên có vô hạn đường đi ⇒ `False`.

In [35]:
hand_answer_solution = False
assert hand_answer_solution == sol(5, [[0, 1], [1, 2], [2, 3], [3, 1], [0, 4]], 0, 4) == leads_brute(5, [[0, 1], [1, 2], [2, 3], [3, 1], [0, 4]], 0, 4)
print("Bài 2 OK")

Bài 2 OK


### Đáp án Bài 3

In [36]:
def exercise_3_solution(n, edges, source, destination):
    graph = [[] for _ in range(n)]
    for a, b in edges:
        graph[a].append(b)
    state, ptr = [0] * n, [0] * n
    state[source] = 1
    stack = [source]
    while stack:
        u = stack[-1]
        if not graph[u]:
            if u != destination:
                return False
            state[u] = 2
            stack.pop()
        elif ptr[u] < len(graph[u]):
            v = graph[u][ptr[u]]
            ptr[u] += 1
            if state[v] == 1:
                return False
            if state[v] == 0:
                state[v] = 1
                stack.append(v)
        else:
            state[u] = 2
            stack.pop()
    return True

for n, e, s, d, ex in examples:
    assert exercise_3_solution(n, e, s, d) == ex
rng = random.Random(3)
for _ in range(1000):
    n = rng.randint(1, 6)
    e = [[rng.randrange(n), rng.randrange(n)] for _ in range(rng.randint(0, 8))]
    s, d = rng.randrange(n), rng.randrange(n)
    assert exercise_3_solution(n, e, s, d) == leads_brute(n, e, s, d)
print("Bài 3 OK")

Bài 3 OK


### Đáp án Bài 4

In [37]:
def exercise_4_solution(n, edges, source, destination):
    graph = [[] for _ in range(n)]
    for a, b in edges:
        graph[a].append(b)
    R, seen = [source], {source}
    for u in R:
        for v in graph[u]:
            if v not in seen:
                seen.add(v)
                R.append(v)
    indeg = {u: 0 for u in R}
    for u in R:
        for v in graph[u]:
            indeg[v] += 1
    queue = deque(u for u in R if indeg[u] == 0)
    processed = 0
    while queue:
        u = queue.popleft()
        processed += 1
        for v in graph[u]:
            indeg[v] -= 1
            if indeg[v] == 0:
                queue.append(v)
    if processed != len(R):
        return False
    return all(u == destination for u in R if not graph[u])

for n, e, s, d, ex in examples:
    assert exercise_4_solution(n, e, s, d) == ex
rng = random.Random(4)
for _ in range(1000):
    n = rng.randint(1, 6)
    e = [[rng.randrange(n), rng.randrange(n)] for _ in range(rng.randint(0, 8))]
    s, d = rng.randrange(n), rng.randrange(n)
    assert exercise_4_solution(n, e, s, d) == leads_brute(n, e, s, d)
print("Bài 4 OK")

Bài 4 OK


### Đáp án Bài 5

In [38]:
def exercise_5_solution(n, edges, source, destination):
    ways = [0] * n
    ways[source] = 1
    for a, b in sorted(edges):                      # sắp theo a tăng dần: ways[a] đã đầy đủ trước khi dùng
        ways[b] += ways[a]
    return ways[destination]

def paths_oracle_solution(n, edges, source, destination):
    graph = [[] for _ in range(n)]
    for a, b in edges:
        graph[a].append(b)
    count, stack = 0, [source]
    while stack:
        u = stack.pop()
        if u == destination:
            count += 1
        stack.extend(graph[u])
    return count

assert exercise_5_solution(4, [[0, 1], [0, 2], [1, 3], [2, 3]], 0, 3) == 2
assert exercise_5_solution(3, [[0, 1], [0, 1], [1, 2]], 0, 2) == 2
assert exercise_5_solution(3, [[0, 1]], 0, 2) == 0
assert exercise_5_solution(1, [], 0, 0) == 1
rng = random.Random(5)
for _ in range(1000):
    n = rng.randint(1, 8)
    e = [[a, rng.randint(a + 1, n - 1)] for a in range(n - 1) for _ in range(rng.randint(0, 2))]
    s, d = rng.randrange(n), rng.randrange(n)
    assert exercise_5_solution(n, e, s, d) == paths_oracle_solution(n, e, s, d), (n, e, s, d)
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

### Template (DFS lặp ba trạng thái)

```python
class Solution:
    def leadsToDestination(self, n: int, edges: List[List[int]], source: int, destination: int) -> bool:
        graph = [[] for _ in range(n)]
        for a, b in edges:
            graph[a].append(b)
        state = [0] * n                      # 0 chưa thăm, 1 đang trên đường đi, 2 đã xong
        ptr = [0] * n
        state[source] = 1
        stack = [source]
        while stack:
            u = stack[-1]
            if not graph[u]:                 # ngõ cụt phải là đích
                if u != destination:
                    return False
                state[u] = 2
                stack.pop()
            elif ptr[u] < len(graph[u]):
                v = graph[u][ptr[u]]
                ptr[u] += 1
                if state[v] == 1:            # chu trình
                    return False
                if state[v] == 0:
                    state[v] = 1
                    stack.append(v)
            else:
                state[u] = 2
                stack.pop()
        return True
```

Công thức nhớ nhanh: **chỉ từ `source`; ngõ cụt phải là `destination`; gặp đỉnh đang trên đường đi là chu trình ⇒ `False`**.

### Checklist trước khi submit

- [ ] Chỉ duyệt các đỉnh đạt được từ `source`?
- [ ] Ngõ cụt khác `destination` ⇒ `False`; `destination` có cạnh ra cũng bị loại (qua kiểm tra ngõ cụt và chu trình)?
- [ ] Phân biệt trạng thái "đang trên đường đi" (`1`, chu trình) và "đã xong" (`2`, kim cương hợp lệ)?
- [ ] Khuyên và cạnh song song được xử lý đúng?
- [ ] Dùng DFS lặp (hoặc nâng giới hạn đệ quy) cho `n = 10^4`?
- [ ] Đã thử: `source = destination`, khuyên ở đích, chu trình không đạt được, kim cương, xích `10^4`?